# [[18, 4, 4]] circuit-level simulation

Stim + `CircuitBuilder` pipeline mirroring `QEC/832/c832.py`:
scheduleable CX syndrome extraction, warmup / noisy rounds / cooldown,
detector postselection, and logical-parity LER on final X readout.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import c1844 as c

print('code [[18, 4, 4]], k =', c.N_DATA - c.gf2_rank(c.hx) - c.gf2_rank(c.hz))
print('lx weights', c.lx.sum(1), 'lz weights', c.lz.sum(1))
print('(lx @ lz.T) % 2 =\n', (c.lx @ c.lz.T) % 2)
print('schedule valid:', c.checkvalidschedule(c.defaultzschedule), c.checkvalidschedule(c.defaultxschedule))
c.defaultzschedule, c.defaultxschedule

In [ ]:
# Zero-noise sanity check
cb = c.c1844scheduleable(er=0, nrounds=1)
sample = cb.get_circuit().compile_sampler().sample(5000)
sl = c.defaultzschedule.shape[0] + c.defaultxschedule.shape[0]
totalshots, totalerrors, survived, survivederrors = c.sim_analysis(sample, sl, 1, c.lx)
print(totalshots, survived, survivederrors, totalerrors)
assert survived == totalshots and all(e == 0 for e in survivederrors)

In [ ]:
nrounds_list = [1, 2, 4]
ers = [1e-4, 1e-3, 1e-2]
nshots = 100_000
sl = c.defaultzschedule.shape[0] + c.defaultxschedule.shape[0]

rows = []
for n in tqdm(nrounds_list):
    for er in ers:
        cb = c.c1844scheduleable(er=er, nrounds=n)
        sample = cb.get_circuit().compile_sampler().sample(nshots)
        totalshots, totalerrors, survived, survivederrors = c.sim_analysis(sample, sl, n, c.lx)
        post = [e / survived if survived else np.nan for e in survivederrors]
        raw = [e / totalshots for e in totalerrors]
        rows.append(dict(nrounds=n, er=er, survived=survived, post=post, raw=raw))
        print(f'n={n} er={er}: survived {survived}/{totalshots}, postsel LER={post}')
rows

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
for n in nrounds_list:
    xs = [r['er'] for r in rows if r['nrounds'] == n]
    # mean postselected LER over the 4 logicals
    ys = [np.nanmean(r['post']) for r in rows if r['nrounds'] == n]
    ax.plot(xs, ys, 'o-', label=f'{n} rounds')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('error rate')
ax.set_ylabel('mean postselected LER')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Optional: inspect Stim diagram for the default schedule
print(c.c1844scheduleable(0.01, 1).get_circuit().diagram())